# Tutorial 3: Model Comparison with Statistical Tests

This tutorial demonstrates how to compare ML, DL, and LLM models using
bootstrap confidence intervals and pairwise statistical tests.

## Available Models

OneEHR supports 25 model architectures:

| Category | Models |
|----------|--------|
| Tabular ML | XGBoost, CatBoost, Random Forest, Decision Tree, GBDT, Logistic Regression |
| Recurrent | GRU, LSTM, RNN, RETAIN, M3Care |
| Non-Recurrent | TCN, Transformer, MLP, Deepr, EHR-Mamba, Jamba |
| EHR-Specialized | AdaCare, StageNet, ConCare, GRASP, MCGRU, DrAgent, PRISM, SAFARI, PAI |
| Survival | DeepSurv, DeepHit |

In [ ]:
# Example config with multiple models
config_text = """
[dataset]
dynamic = "examples/tjh/dynamic.csv"
label = "examples/tjh/label_mortality.csv"

[preprocess]
bin_size = "1d"
top_k_codes = 50

[task]
kind = "binary"
prediction_mode = "patient"

[split]
kind = "random"
seed = 42
val_size = 0.15
test_size = 0.15

# ML models
[[models]]
name = "xgboost"
[models.params]
max_depth = 4
n_estimators = 100

[[models]]
name = "lr"

# DL models
[[models]]
name = "gru"

[[models]]
name = "retain"

[[models]]
name = "transformer"

[trainer]
device = "cpu"
seed = 42
max_epochs = 20
batch_size = 32
lr = 0.001
early_stopping = true
patience = 5

[output]
root = "runs"
run_name = "tutorial_comparison"
"""

print("Config defines 5 models: xgboost, lr, gru, retain, transformer")

## Statistical Comparison Framework

OneEHR automatically computes:
- **Bootstrap CI** (200 resamples by default) for AUROC, AUPRC, F1
- **DeLong test** for pairwise AUROC comparison
- **McNemar test** for classification error comparison
- **BH FDR correction** for multiple comparisons

In [ ]:
# Demonstrate the statistical testing module
import numpy as np
import pandas as pd
from oneehr.analysis.statistical_tests import compute_statistical_tests

# Simulate predictions from 3 models
rng = np.random.default_rng(42)
n = 200
pids = [f"p{i}" for i in range(n)]
y_true = rng.integers(0, 2, size=n).astype(float)

preds = pd.concat([
    pd.DataFrame({
        "patient_id": pids, "system": "xgboost",
        "y_true": y_true, "y_pred": np.clip(y_true + rng.normal(0, 0.3, n), 0, 1),
    }),
    pd.DataFrame({
        "patient_id": pids, "system": "gru",
        "y_true": y_true, "y_pred": np.clip(y_true + rng.normal(0, 0.35, n), 0, 1),
    }),
    pd.DataFrame({
        "patient_id": pids, "system": "retain",
        "y_true": y_true, "y_pred": np.clip(y_true + rng.normal(0, 0.4, n), 0, 1),
    }),
], ignore_index=True)

result = compute_statistical_tests(preds=preds)
print(f"Pairwise comparisons: {len(result['pairwise'])}")
for pair in result['pairwise']:
    print(f"  {pair['system_a']} vs {pair['system_b']}:")
    print(f"    DeLong p={pair['delong']['p_value']:.4f}")
    print(f"    McNemar p={pair['mcnemar']['p_value']:.4f}")

In [ ]:
# Bootstrap confidence intervals
from oneehr.eval.bootstrap import bootstrap_metric
from oneehr.config.schema import TaskConfig

for sys_name in ["xgboost", "gru", "retain"]:
    sys_preds = preds[preds["system"] == sys_name]
    result = bootstrap_metric(
        y_true=sys_preds["y_true"].values,
        y_pred=sys_preds["y_pred"].values,
        task=TaskConfig(kind="binary"),
        metric="auroc",
        n=200,
    )
    print(f"{sys_name}: AUROC = {result.mean:.4f} [{result.ci_low:.4f}, {result.ci_high:.4f}]")

## Including LLM Agents

OneEHR can also run LLM-based predictions alongside trained models:

```toml
[[systems]]
name = "gpt4o_agent"
kind = "llm"
framework = "single_llm"
backend = "openai"
model = "gpt-4o"
api_key_env = "OPENAI_API_KEY"
```

All predictions (ML + DL + LLM) are unified in `predictions.parquet` with a `system` column for fair comparison.